In [0]:
CREATE SCHEMA IF NOT EXISTS hackaton.silver_data;



In [0]:
-- ciao a tutti 

In [0]:
CREATE OR REPLACE TABLE hackaton.silver_data.silver_customers AS
WITH extracted_customers AS (
  SELECT 
    trim(split(customer_info, '\\|')[1]) AS customer_email,
    trim(split(customer_info, '\\|')[0]) AS customer_name,
    trim(split(customer_info, '\\|')[2]) AS customer_phone
  FROM hackaton.bronze_data.bronze_sales
  WHERE customer_info IS NOT NULL AND contains(customer_info, '|')
)
SELECT 
  customer_email,
  FIRST_VALUE(customer_name, true) OVER (PARTITION BY customer_email ORDER BY customer_name DESC) AS customer_name,
  FIRST_VALUE(customer_phone, true) OVER (PARTITION BY customer_email ORDER BY customer_phone DESC) AS customer_phone
FROM extracted_customers
WHERE customer_email IS NOT NULL AND trim(lower(customer_email)) NOT IN ('null', '')
GROUP BY customer_email, customer_name, customer_phone;

SELECT * FROM hackaton.silver_data.silver_customers;

In [0]:
CREATE OR REPLACE TABLE hackaton.silver_data.silver_sales_transactions AS
WITH cleaned_sales AS (
  SELECT 
    order_id,
    
    -- Parsing date sicuro con try_to_date per gestire tutti i formati
    COALESCE(
      CASE WHEN lower(trim(transaction_date)) = 'today' THEN current_date() ELSE NULL END,
      try_to_date(transaction_date, 'dd/MM/yyyy'),
      try_to_date(transaction_date, 'yyyy-MM-dd'),
      try_to_date(transaction_date, 'dd-MMM-yyyy'),
      try_to_date(transaction_date, 'MM/dd/yyyy'),
      try_to_date(transaction_date, 'yyyy/MM/dd')
    ) AS transaction_date_clean,

    -- FK di raccordo verso silver_customers
    trim(split(customer_info, '\\|')[1]) AS customer_email,
    
    product_id,
    product_category,

    -- Pulizia prezzo (rimuove $, €, £, spazi e virgole)
    CAST(regexp_replace(price, '[\\$,€,£,\\s,]', '') AS DOUBLE) AS price_clean,

    -- Quantità (mantiene negativi per i resi)
    CAST(qty AS INT) AS qty_clean,

    -- Standardizzazione sconto
    CASE 
      WHEN contains(discount_pct, '%') THEN CAST(regexp_replace(discount_pct, '%', '') AS DOUBLE) / 100.0
      WHEN discount_pct IN ('N/A', 'null', '') OR discount_pct IS NULL THEN 0.0
      ELSE CAST(discount_pct AS DOUBLE)
    END AS discount_clean

  FROM hackaton.bronze_data.bronze_sales
)
SELECT 
  order_id,
  COALESCE(transaction_date_clean, current_date()) AS transaction_date,
  customer_email,
  product_id,
  product_category,
  price_clean AS unit_price,
  qty_clean AS qty,
  discount_clean AS discount_pct,
  ROUND((qty_clean * price_clean * (1 - discount_clean)),2) AS net_revenue
FROM cleaned_sales;

SELECT * FROM hackaton.silver_data.silver_sales_transactions;

In [0]:
CREATE OR REPLACE TABLE hackaton.silver_data.silver_web_reviews AS
WITH clean_strings AS (
  SELECT 
    review_id,
    product_ref AS product_id,
    email AS customer_email,
    country,
    city,
    review_text,
    CASE WHEN trim(lower(timestamp)) IN ('null', '') THEN NULL ELSE timestamp END AS raw_ts,
    CASE WHEN trim(lower(rating)) IN ('null', '') THEN NULL ELSE rating END AS raw_rating
  FROM hackaton.bronze_data.bronze_web_reviews
),
parsed_numbers AS (
  SELECT
    review_id,
    product_id,
    customer_email,
    country,
    city,
    review_text,
    try_cast(raw_ts AS BIGINT) AS ts_bigint,
    try_cast(raw_rating AS INT) AS rating_int
  FROM clean_strings
)
SELECT 
  review_id,
  product_id,
  customer_email,
  country,
  city,
  
  -- Conversione Epoch Timestamp in data formattata YYYY-MM-DD
  CASE 
    WHEN ts_bigint IS NULL THEN NULL
    WHEN ts_bigint > 9999999999 THEN to_date(from_unixtime(ts_bigint / 1000))
    ELSE to_date(from_unixtime(ts_bigint))
  END AS review_date,

  -- Rating pulito (mantiene solo valori 1-5, altrimenti NULL)
  CASE 
    WHEN rating_int BETWEEN 1 AND 5 THEN rating_int
    ELSE NULL
  END AS rating_clean,

  review_text
FROM parsed_numbers;

SELECT * FROM hackaton.silver_data.silver_web_reviews;


In [0]:
CREATE OR REPLACE TABLE hackaton.silver_data.silver_finance_targets AS
SELECT 
  Region AS region,
  Category AS product_category,
  -- Conversione stringa mese dalla colonna ColumnNames in formato DATE (YYYY-MM-DD)
  to_date(ColumnNames, 'MMM-yy') AS target_date,
  -- Target Finanziario a 2 decimali dalla colonna ColumnValues
  ROUND(CAST(ColumnValues AS DOUBLE), 2) AS target_amount
FROM hackaton.bronze_data.bronze_finance_targets;   

SELECT * FROM hackaton.silver_data.silver_finance_targets;

In [0]:
CREATE SCHEMA IF NOT EXISTS hackaton.gold_data;

In [0]:
CREATE OR REPLACE TABLE hackaton.gold_data.gold_dim_customers AS
SELECT 
  row_number() OVER (ORDER BY customer_email) AS customer_key,
  customer_email,
  customer_name,
  customer_phone
FROM hackaton.silver_data.silver_customers;

SELECT * FROM hackaton.gold_data.gold_dim_customers;


In [0]:
CREATE OR REPLACE TABLE hackaton.gold_data.gold_dim_products AS
WITH unk_products AS (
  SELECT product_id, product_category
  FROM hackaton.silver_data.silver_sales_transactions
  WHERE product_id IS NOT NULL
  
  UNION ALL
  
  SELECT product_id, NULL AS product_category
  FROM hackaton.silver_data.silver_web_reviews
  WHERE product_id IS NOT NULL
),
consolidated AS (
  SELECT 
    product_id,
    COALESCE(
      FIRST_VALUE(product_category, true) OVER (PARTITION BY product_id ORDER BY product_category DESC), 
      'Uncategorized'
    ) AS product_category
  FROM unk_products
)
SELECT DISTINCT 
  product_id, 
  product_category
FROM consolidated;
SELECT * FROM hackaton.gold_data.gold_dim_products;


In [0]:
CREATE OR REPLACE TABLE hackaton.gold_data.gold_dim_date AS
WITH date_range AS (
  SELECT min(transaction_date) AS min_date, max(transaction_date) AS max_date 
  FROM hackaton.silver_data.silver_sales_transactions
),
exploded_dates AS (
  SELECT explode(sequence(min_date, max_date, interval 1 day)) AS full_date
  FROM date_range
)
SELECT 
  full_date AS date_key,
  year(full_date) AS year,
  month(full_date) AS month,
  date_format(full_date, 'MMMM') AS month_name,
  quarter(full_date) AS quarter,
  dayofweek(full_date) AS day_of_month
FROM exploded_dates;

SELECT * FROM hackaton.gold_data.gold_dim_date;

In [0]:
CREATE OR REPLACE TABLE hackaton.gold_data.gold_fact_sales AS
SELECT 
  s.order_id,
  s.transaction_date AS date_key,
  c.customer_key,
  s.product_id,
  s.qty,
  s.unit_price,
  s.discount_pct,
  s.net_revenue
FROM hackaton.silver_data.silver_sales_transactions s
LEFT JOIN hackaton.gold_data.gold_dim_customers c 
  ON s.customer_email = c.customer_email;

  SELECT * FROM hackaton.gold_data.gold_fact_sales;


In [0]:
CREATE OR REPLACE TABLE hackaton.gold_data.gold_fact_targets AS
SELECT 
  row_number() OVER (ORDER BY target_date, product_category, region) AS target_id,
  target_date AS date_key,
  product_category,
  region,
  target_amount
FROM hackaton.silver_data.silver_finance_targets;

SELECT * FROM hackaton.gold_data.gold_fact_targets;

In [0]:
CREATE OR REPLACE TABLE hackaton.gold_data.gold_dim_category AS
SELECT DISTINCT 
  product_category
FROM hackaton.gold_data.gold_dim_products
WHERE product_category IS NOT NULL;

SELECT * FROM hackaton.gold_data.gold_dim_category;